# Recipe Ratings and Ingredients Analysis

This notebook presents a short descriptive analysis of two datasets:

- `allrecipes_all.csv`
- `recipes_ingredients_long.csv`

The goal is to:

- inspect the structure of the datasets
- clean key nutrition variables
- create a recipe-level summary
- visualize the main patterns
- identify ingredients associated with lower ratings


## 1. Import Libraries and Load Data

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import re

# Load data
allrecipes_df = pd.read_csv("allrecipes_all.csv")
ingredients_df = pd.read_csv("recipes_ingredients_long.csv")

print("AllRecipes shape:", allrecipes_df.shape)
print("Ingredients shape:", ingredients_df.shape)


AllRecipes shape: (14438, 25)
Ingredients shape: (142466, 13)


## 2. Quick Overview

This section gives a quick summary of both datasets:

- shape
- columns
- first rows
- missing values
- main numerical statistics


In [3]:
print("ALLRECIPES DATASET")
print("Shape:", allrecipes_df.shape)
print("Columns:", allrecipes_df.columns.tolist())

print("\nFirst rows:")
print(allrecipes_df.head())

print("\nMissing values (%):")
print((allrecipes_df.isna().mean() * 100).round(1).sort_values(ascending=False).head(10))

print("\nMain numerical summary:")
print(allrecipes_df[["rating_value", "rating_count", "review_count"]].describe())


ALLRECIPES DATASET
Shape: (14438, 25)
Columns: ['author', 'category', 'cook_time', 'cuisine', 'date_published', 'description', 'directions', 'ingredients', 'letter', 'nutrition_calories', 'nutrition_carbs', 'nutrition_fat', 'nutrition_protein', 'nutrition_raw', 'prep_time', 'rating_count', 'rating_text', 'rating_value', 'review_count', 'source_category', 'source_category_url', 'title', 'total_time', 'url', 'yield']

First rows:
              author                      category cook_time       cuisine  \
0     Nicole Russell          ['Dinner', 'Entree']     PT15M  ['American']   
1     Nicole Russell       ['Appetizer', 'Dinner']     PT10M  ['American']   
2  Nicole McLaughlin       ['Appetizer', 'Dinner']     PT10M   ['Mexican']   
3  Nicole McLaughlin  ['Snack', 'Lunch', 'Entree']      PT5M  ['American']   
4  Yolanda Gutierrez       ['Dinner', 'Side Dish']     PT15M  ['American']   

              date_published  \
0  2025-06-05T09:15:00-04:00   
1  2024-02-23T19:16:13-05:00   
2  

In [4]:
print("INGREDIENTS DATASET")
print("Shape:", ingredients_df.shape)
print("Columns:", ingredients_df.columns.tolist())

print("\nFirst rows:")
print(ingredients_df.head())

print("\nMissing values (%):")
print((ingredients_df.isna().mean() * 100).round(1).sort_values(ascending=False).head(10))

print("\nMain numerical summary:")
print(ingredients_df[["rating_value", "rating_count"]].describe())


INGREDIENTS DATASET
Shape: (142466, 13)
Columns: ['recipe_id', 'title', 'ingredient_index', 'ingredient_raw', 'ingredient_clean', 'ingredient_canonical', 'rating_value', 'rating_count', 'nutrition_calories', 'nutrition_carbs', 'nutrition_fat', 'nutrition_protein', 'url']

First rows:
   recipe_id                                              title  \
0          0  3-Ingredient Air Fryer Everything Bagel Chicke...   
1          0  3-Ingredient Air Fryer Everything Bagel Chicke...   
2          0  3-Ingredient Air Fryer Everything Bagel Chicke...   
3          1              4 Ingredient Air Fryer Pepper Poppers   
4          1              4 Ingredient Air Fryer Pepper Poppers   

   ingredient_index                      ingredient_raw  \
0                 1   1 1/4 pound fresh chicken tenders   
1                 2              1 tablespoon olive oil   
2                 3  1/3 cup everything bagel seasoning   
3                 1            1 bell pepper, any color   
4                

## 3. Basic Cleaning

Nutrition variables are stored as text (for example, `"366 kcal"` or `"22 g"`).  
The function below extracts the numeric value and creates cleaned numeric columns.


In [5]:
def extract_number(value):
    if pd.isna(value):
        return np.nan

    value = str(value).replace(",", "")
    match = re.search(r"[-+]?\d*\.?\d+", value)

    if match:
        return float(match.group())
    return np.nan


# Convert nutrition columns to numeric
for col in ["nutrition_calories", "nutrition_carbs", "nutrition_fat", "nutrition_protein"]:
    allrecipes_df[col + "_num"] = allrecipes_df[col].apply(extract_number)
    ingredients_df[col + "_num"] = ingredients_df[col].apply(extract_number)


## 4. Recipe-Level Summary

The ingredient dataset contains multiple rows per recipe.  
To analyze recipes more easily, we aggregate to one row per recipe with:

- number of unique ingredients
- rating value
- rating count
- calories
- carbs
- fat
- protein


In [6]:
recipe_summary = (
    ingredients_df
    .groupby(["recipe_id", "title", "url"], as_index=False)
    .agg(
        ingredient_count=("ingredient_canonical", "nunique"),
        rating_value=("rating_value", "first"),
        rating_count=("rating_count", "first"),
        calories=("nutrition_calories_num", "first"),
        carbs=("nutrition_carbs_num", "first"),
        fat=("nutrition_fat_num", "first"),
        protein=("nutrition_protein_num", "first")
    )
)

print("RECIPE-LEVEL SUMMARY")
print("Shape:", recipe_summary.shape)

print("\nSummary statistics:")
print(recipe_summary[["ingredient_count", "rating_value", "rating_count", "calories"]].describe())

print("\nCorrelation with rating:")
print(
    recipe_summary[
        ["rating_value", "rating_count", "ingredient_count", "calories", "carbs", "fat", "protein"]
    ]
    .corr()["rating_value"]
    .sort_values(ascending=False)
)


RECIPE-LEVEL SUMMARY
Shape: (14396, 10)

Summary statistics:
       ingredient_count  rating_value  rating_count      calories
count      14396.000000  13516.000000  14396.000000  14200.000000
mean           9.671992      4.529609    210.093081    349.117042
std            3.962723      0.411447    751.828205    254.741234
min            1.000000      1.000000      0.000000      1.000000
25%            7.000000      4.400000      4.000000    183.000000
50%            9.000000      4.600000     27.000000    309.500000
75%           12.000000      4.800000    131.000000    458.000000
max           32.000000      5.000000  20956.000000   9538.000000

Correlation with rating:
rating_value        1.000000
rating_count        0.068867
ingredient_count    0.065652
fat                 0.054398
calories            0.040124
protein             0.025961
carbs               0.002186
Name: rating_value, dtype: float64


## 5. Main Plots

These plots show the main descriptive patterns in the data.


In [7]:
fig = px.histogram(
    allrecipes_df,
    x="rating_value",
    nbins=30,
    title="Distribution of Recipe Ratings"
)
fig.show()


In [8]:
top_ingredients = (
    ingredients_df["ingredient_canonical"]
    .value_counts()
    .head(20)
    .reset_index()
)

top_ingredients.columns = ["ingredient", "count"]

fig = px.bar(
    top_ingredients,
    x="ingredient",
    y="count",
    title="Top 20 Most Frequent Ingredients"
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()


In [9]:
fig = px.histogram(
    recipe_summary,
    x="ingredient_count",
    nbins=30,
    title="Distribution of Number of Ingredients per Recipe"
)
fig.show()


In [10]:
fig = px.scatter(
    recipe_summary,
    x="ingredient_count",
    y="rating_value",
    size="rating_count",
    hover_name="title",
    title="Rating vs Number of Ingredients",
    opacity=0.5
)
fig.show()


In [11]:
corr_cols = [
    "rating_value",
    "rating_count",
    "ingredient_count",
    "calories",
    "carbs",
    "fat",
    "protein"
]

corr_matrix = recipe_summary[corr_cols].corr()

fig = px.imshow(
    corr_matrix,
    text_auto=True,
    title="Correlation Heatmap"
)
fig.show()


## 6. Ingredients Associated with Low Ratings

A recipe is considered low-rated if:

```python
rating_value < 4.0
```

To reduce noise, we keep only ingredients that appear in at least 30 recipes.


In [12]:
low_rating_threshold = 4.0
min_recipe_count = 30

ingredient_rating = ingredients_df.copy()
ingredient_rating["is_low_rated"] = ingredient_rating["rating_value"] < low_rating_threshold

ingredient_summary = (
    ingredient_rating
    .groupby("ingredient_canonical", as_index=False)
    .agg(
        recipe_count=("recipe_id", "nunique"),
        avg_rating=("rating_value", "mean"),
        low_rated_recipes=("is_low_rated", "sum")
    )
)

ingredient_summary["low_rating_share"] = (
    ingredient_summary["low_rated_recipes"] / ingredient_summary["recipe_count"]
)

ingredient_summary = ingredient_summary[
    ingredient_summary["recipe_count"] >= min_recipe_count
].sort_values("avg_rating")

print("INGREDIENTS WITH LOWEST AVERAGE RATINGS")
print(ingredient_summary.head(15))


INGREDIENTS WITH LOWEST AVERAGE RATINGS
                                   ingredient_canonical  recipe_count  \
17962                                plain greek yogurt            37   
20341                                         skim milk            42   
9404                                  cooked white rice            37   
22749                               uncooked white rice            80   
6990                                        black olive            52   
5991                                        almond milk            41   
8917                                       cocoa powder            45   
9473                                         corn syrup            38   
14828                                     lemon extract            42   
1125   10.75 ounce can condensed cream of mushroom soup            77   
6178                                         applesauce            42   
23455                               white sugar or more            40   
7455       

In [13]:
fig = px.bar(
    ingredient_summary.head(15),
    x="ingredient_canonical",
    y="avg_rating",
    hover_data=["recipe_count", "low_rating_share"],
    title="Ingredients with Lowest Average Ratings"
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()


## Conclusion

This notebook provides a concise descriptive analysis of recipe ratings and ingredients.

It highlights:

- the structure and quality of the datasets
- the distribution of ratings
- the most frequent ingredients
- simple correlations between recipe characteristics and ratings
- ingredients that are associated with lower average ratings
